In [2]:
!pip install xgboost
import pandas as pd
import numpy as np

from google.colab import files

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Regression models
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor

In [3]:
uploaded = files.upload()  # Choose your dataset file from your device

# Replace with the exact filename if needed
filename = list(uploaded.keys())[0]

df = pd.read_csv(filename)
print("Shape:", df.shape)
df.head()

Saving student_data.csv to student_data.csv
Shape: (30, 13)


,gender,age,study_time,absences,famrel,freetime,health,activities,failures,G1,G2,G3,performance
0,F,17,2,4,4,3,3,yes,0,13,14,15,High
1,M,18,3,10,3,4,5,no,0,10,11,12,Medium
2,F,16,1,2,5,2,3,yes,0,15,15,16,High
3,M,16,2,6,4,3,4,yes,1,8,9,9,Low
4,F,17,3,0,5,4,3,yes,0,14,15,15,High


In [4]:
df.info()
df.describe(include='all')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 13 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   gender       30 non-null     object
 1   age          30 non-null     int64 
 2   study_time   30 non-null     int64 
 3   absences     30 non-null     int64 
 4   famrel       30 non-null     int64 
 5   freetime     30 non-null     int64 
 6   health       30 non-null     int64 
 7   activities   30 non-null     object
 8   failures     30 non-null     int64 
 9   G1           30 non-null     int64 
 10  G2           30 non-null     int64 
 11  G3           30 non-null     int64 
 12  performance  30 non-null     object
dtypes: int64(10), object(3)
memory usage: 3.2+ KB


,gender,age,study_time,absences,famrel,freetime,health,activities,failures,G1,G2,G3,performance
count,30,30.000000,30.000000,30.00000,30.000000,30.000000,30.000000,30,30.000000,30.000000,30.000000,30.000000,30
unique,2,NaN,NaN,NaN,NaN,NaN,NaN,2,NaN,NaN,NaN,NaN,3
top,F,NaN,NaN,NaN,NaN,NaN,NaN,yes,NaN,NaN,NaN,NaN,High
freq,15,NaN,NaN,NaN,NaN,NaN,NaN,18,NaN,NaN,NaN,NaN,11
mean,NaN,16.866667,2.033333,7.40000,3.700000,3.166667,3.533333,NaN,0.800000,10.833333,11.566667,11.766667,NaN
std,NaN,0.937102,0.808717,6.24555,1.118805,0.874281,0.973204,NaN,0.996546,3.238596,3.223655,3.480818,NaN
min,NaN,15.000000,1.000000,0.00000,2.000000,2.000000,2.000000,NaN,0.000000,5.000000,6.000000,5.000000,NaN
25%,NaN,16.000000,1.000000,2.00000,3.000000,3.000000,3.000000,NaN,0.000000,8.000000,8.250000,9.000000,NaN
50%,NaN,17.000000,2.000000,6.00000,4.000000,3.000000,4.000000,NaN,0.000000,11.000000,12.000000,12.000000,NaN
75%,NaN,18.000000,3.000000,11.50000,5.000000,4.000000,4.000000,NaN,1.750000,14.000000,14.750000,15.000000,NaN


In [6]:
TARGET_COL = "G3"   # 🔴 CHANGE THIS if your column name is different

if TARGET_COL not in df.columns:
    raise ValueError(f"Target column '{TARGET_COL}' not found. "
                     "Change TARGET_COL to your actual final grade column (e.g. 'G3').")

X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

# Drop rows where target is missing
mask = ~y.isna()
X = X[mask]
y = y[mask]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (30, 12)
Target shape: (30,)


In [7]:
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object', 'bool', 'category']).columns.tolist()

print("Numeric columns:", numeric_features)
print("Categorical columns:", categorical_features)

Numeric columns: ['age', 'study_time', 'absences', 'famrel', 'freetime', 'health', 'failures', 'G1', 'G2']
Categorical columns: ['gender', 'activities', 'performance']


In [10]:
numeric_transformer = Pipeline(steps=[
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train shape:", X_train.shape, y_train.shape)
print("Test shape:", X_test.shape, y_test.shape)

Train shape: (24, 12) (24,)
Test shape: (6, 12) (6,)


In [13]:
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(
        n_estimators=200, random_state=42
    ),
    "XGBoost": XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        objective="reg:squarederror"
    ),
    "Support Vector Regressor (SVR)": SVR()
}

results = []

for name, model in models.items():
    print(f"\n==================== {name} ====================")

    # Create full pipeline: preprocessing + model
    clf = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    # Train
    clf.fit(X_train, y_train)

    # Predict
    y_pred = clf.predict(X_test)

    # Evaluation metrics
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)

    results.append((name, mae, rmse, r2))

    print(f"MAE  : {mae:.3f}")
    print(f"RMSE : {rmse:.3f}")
    print(f"R²   : {r2:.3f}")


==================== Linear Regression ====================
MAE  : 0.744
RMSE : 0.885
R²   : 0.875

==================== Random Forest ====================
MAE  : 0.452
RMSE : 0.610
R²   : 0.941

==================== XGBoost ====================
MAE  : 0.555
RMSE : 0.717
R²   : 0.918

==================== Support Vector Regressor (SVR) ====================
MAE  : 0.487
RMSE : 0.626
R²   : 0.937


In [14]:
results_df = pd.DataFrame(
    results,
    columns=["Model", "MAE", "RMSE", "R2 Score"]
)
results_df.sort_values(by="R2 Score", ascending=False)

,Model,MAE,RMSE,R2 Score
1,Random Forest,0.451667,0.609570,0.940548
3,Support Vector Regressor (SVR),0.486765,0.626273,0.937245
2,XGBoost,0.554792,0.716933,0.917761
0,Linear Regression,0.744389,0.884539,0.874814


In [15]:
# Example: choose Random Forest as best (or pick based on results_df)
best_model = RandomForestRegressor(
    n_estimators=200, random_state=42
)

best_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", best_model)
])

best_pipeline.fit(X, y)

import joblib
joblib.dump(best_pipeline, "student_grade_predictor.pkl")

print("Model saved as: student_grade_predictor.pkl")

Model saved as: student_grade_predictor.pkl
